In [16]:
import os
import pickle
import numpy as np
np.random.seed(42)
import pandas as pd
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

In [17]:
dataset = "LastFM"
group = "tail"
sim_metric = "cos"
topk = 100

In [18]:
# load the llm item embedding
item_emb = pickle.load(open(os.path.join(dataset+"/handled/"+group+"_item_emb_np.pkl"), "rb"))
print(f"{group} item embedding shape:", item_emb.shape)

tail item embedding shape: (4872, 1536)


In [19]:
# calculate the similarity score between users based on llm user embedding
from sklearn.cluster import KMeans
X = item_emb.astype(np.float32, copy=False)
norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
X = X / norms
N = X.shape[0]
num_bins = max(32, N // 500)
km = KMeans(n_clusters=num_bins, random_state=42, n_init=10)
labels = km.fit_predict(X)
final_rank_matrix = np.empty((N, topk), dtype=np.int32)
for c in range(num_bins):
    idx = np.where(labels == c)[0]
    if idx.size == 0:
        continue
    Xi = X[idx]
    S = Xi @ Xi.T
    order = np.argsort(-S, axis=1)
    local = order[:, 1:1+topk]
    final_rank_matrix[idx] = idx[local]

/home/dell/anaconda3/envs/llm/lib/python3.9/site-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (32). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [20]:
pass

In [21]:
rank_matrix = None
print("rank matrix shape:", (N, N))

rank matrix shape: (4872, 4872)


In [22]:
final_rank_matrix = final_rank_matrix
print("final rank matrix shape:", final_rank_matrix.shape)

final rank matrix shape: (4872, 100)


In [23]:
print(final_rank_matrix[0])

[3251 3250 3249 3248 3247 3246 3245 3244 3243 3242 3241 3240 3239 3238
 3237 3252 3236 3253 3255 3270 3269 3268 3267 3266 3265 3264 3263 3262
 3261 3260 3259 3258 3257 3256 3254 3271 3235 3233 3213 3212 3211 3210
 3209 3208 3207 3206 3205 3204 3203 3202 3201 3200 3199 3214 3234 3215
 3217 3232 3231 3230 3229 3228 3227 3226 3225 3224 3223 3222 3221 3220
 3219 3218 3216 3198 3272 3274 3327 3326 3325 3324 3323 3322 3321 3320
 3319 3318 3317 3316 3315 3314 3313 3328 3312 3329 3331 3346 3345 3344
 3343 3342]


In [24]:
## Save llm embedding based similar users
pickle.dump(final_rank_matrix, open(os.path.join(dataset+"/handled/"+group+"_sim_item_"+str(topk)+".pkl"), "wb"))